In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import joblib

RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")
REPORTS = Path("../Reports")
REPORTS.mkdir(parents=True, exist_ok=True)

In [2]:
# Metadata
df_meta = pd.read_csv(RAW / "games_detailed_info.csv", low_memory=False)

# Make sure we have an id index
if "id" not in df_meta.columns:
    raise ValueError("games_detailed_info.csv must contain column 'id'")

df_meta = df_meta.set_index("id")

# Ensure name column exists
if "name" not in df_meta.columns:
    if "primary" in df_meta.columns:
        df_meta["name"] = df_meta["primary"]
    elif "Name" in df_meta.columns:
        df_meta["name"] = df_meta["Name"]
    else:
        df_meta["name"] = ""

meta_ids = df_meta.index.astype(int).to_numpy()
id_to_idx = {int(gid): i for i, gid in enumerate(meta_ids)}
idx_to_id = {i: int(gid) for i, gid in enumerate(meta_ids)}

# Similarity matrices (must exist)
desc_sim = joblib.load(PROCESSED / "desc_topk_csr.joblib")
content_sim = joblib.load(PROCESSED / "content_topk_csr.joblib")
emb_sim = joblib.load(PROCESSED / "emb_topk_csr.joblib")

print("Items in meta:", len(meta_ids))

Items in meta: 21631


In [3]:
sent_path = PROCESSED / "sentiment_summary.csv"
sent_summary = pd.read_csv(sent_path, index_col="id") if sent_path.exists() else None

sent_mean = None
if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
    sent_mean = float(sent_summary["avg_sentiment_score"].mean())

print("Sentiment loaded:", sent_summary is not None)

Sentiment loaded: True


In [4]:
ratings_path = RAW / "large_user_ratings.csv"
ratings_df = pd.read_csv(ratings_path)

USER_COL = "Username"
ITEM_COL = "BGGId"
RATING_COL = "Rating"

ratings_df = ratings_df.dropna(subset=[USER_COL, ITEM_COL, RATING_COL]).copy()
ratings_df[ITEM_COL] = ratings_df[ITEM_COL].astype(int)

pop_counts = ratings_df.groupby(ITEM_COL).size()
pop_counts = pop_counts.reindex(meta_ids).fillna(0).astype(int)

# normalized popularity 0..1
pop_norm = (pop_counts - pop_counts.min()) / (pop_counts.max() - pop_counts.min() + 1e-9)
pop_norm = pop_norm.fillna(0.0)

print("Popularity computed.")

Popularity computed.


In [5]:
def _to_tokens(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    s = str(x).strip()
    if not s:
        return []
    # common formats: "['A', 'B']" or "A|B|C" or "A, B, C"
    s = s.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    # choose delimiter
    if "|" in s:
        parts = s.split("|")
    elif "," in s:
        parts = s.split(",")
    else:
        parts = [s]
    return [p.strip() for p in parts if p.strip()]

def get_tokens_col(df, col):
    if col not in df.columns:
        return pd.Series([[]]*len(df), index=df.index)
    return df[col].apply(_to_tokens)

In [6]:
# Try common column names (you can adjust later if needed)
MECH_COLS = ["mechanics", "mechanic", "mechanics_name", "mechanics_names"]
THEME_COLS = ["themes", "theme", "themes_name", "themes_names"]
PUB_COLS  = ["publisher", "publishers", "publisher_name", "publishers_name"]

def first_existing(cols):
    for c in cols:
        if c in df_meta.columns:
            return c
    return None

MECH_COL = "boardgamemechanic"
THEME_COL = "boardgamecategory"
FAMILY_COL = "boardgamefamily"
PUB_COL  = "boardgamepublisher"

mech_tokens = get_tokens_col(df_meta, MECH_COL)
theme_tokens = get_tokens_col(df_meta, THEME_COL)
family_tokens = get_tokens_col(df_meta, FAMILY_COL)
pub_tokens = get_tokens_col(df_meta, PUB_COL)

print("Using columns:")
print("MECH_COL =", MECH_COL)
print("THEME_COL =", THEME_COL)
print("FAMILY_COL =", FAMILY_COL)
print("PUB_COL  =", PUB_COL)

df_meta.loc[[13, 30549, 822], [MECH_COL, THEME_COL, FAMILY_COL, PUB_COL]]

Using columns:
MECH_COL = boardgamemechanic
THEME_COL = boardgamecategory
FAMILY_COL = boardgamefamily
PUB_COL  = boardgamepublisher


,boardgamemechanic,boardgamecategory,boardgamefamily,boardgamepublisher
id,,,,
13,"['Dice Rolling', 'Hexagon Grid', 'Income', 'Mo...","['Economic', 'Negotiation']","['Animals: Sheep', 'Components: Hexagonal Tile...","['KOSMOS', '999 Games', 'Albi', 'Asmodee', 'As..."
30549,"['Action Points', 'Cooperative Game', 'Hand Ma...",['Medical'],"['Components: Map (Global Scale)', 'Components...","['Z-Man Games', 'Albi', 'Asmodee', 'Asmodee It..."
822,"['Area Majority / Influence', 'Map Addition', ...","['City Building', 'Medieval', 'Territory Build...","['Cities: Carcassonne (France)', 'Components: ...","['Hans im Glück', '999 Games', 'Albi', 'Bard C..."


In [7]:
def topk_from_sim(seed_ids, sim_csr, k=800, strategy="mean"):
    seed_idxs = [id_to_idx[sid] for sid in seed_ids if sid in id_to_idx]
    if not seed_idxs:
        return pd.Series(dtype=float)

    scores = defaultdict(float)
    counts = defaultdict(int)

    for si in seed_idxs:
        row = sim_csr.getrow(si)
        for c, v in zip(row.indices, row.data):
            gid = idx_to_id[c]
            if strategy == "max":
                scores[gid] = max(scores[gid], float(v))
            else:
                scores[gid] += float(v)
                counts[gid] += 1

    if strategy != "max":
        for gid in list(scores.keys()):
            scores[gid] /= max(counts[gid], 1)

    for sid in seed_ids:
        scores.pop(int(sid), None)

    items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.Series(dict(items), dtype=float)

def safe_minmax(series, full_index):
    s = series.reindex(full_index).astype(float)
    arr = s.values
    if np.all(np.isnan(arr)):
        return pd.Series(0.0, index=full_index)
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if mx == mn:
        return pd.Series(0.0, index=full_index)
    arr = np.nan_to_num(arr, nan=np.nanmean(arr))
    norm = (arr - mn) / (mx - mn + 1e-9)
    return pd.Series(norm, index=full_index)

In [8]:
def recommend_like(seed_game_ids, topk=20,
                   w_text=0.35, w_emb=0.35, w_sent=0.10, w_pop=0.20,
                   candidate_k=1200):
    seed_game_ids = [int(x) for x in seed_game_ids if int(x) in id_to_idx]
    if len(seed_game_ids) == 0:
        return pd.DataFrame(columns=["game_id","name","score"])

    # Similarity components
    desc_s = topk_from_sim(seed_game_ids, desc_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    cont_s = topk_from_sim(seed_game_ids, content_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    emb_s  = topk_from_sim(seed_game_ids, emb_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)

    text_norm = safe_minmax(0.5*desc_s + 0.5*cont_s, meta_ids)
    emb_norm  = safe_minmax(emb_s, meta_ids)

    # Sentiment
    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(meta_ids).fillna(sent_mean)
        sent_norm = safe_minmax(sent, meta_ids)
    else:
        sent_norm = pd.Series(0.0, index=meta_ids)

    # Popularity (already normalized 0..1)
    pop = pop_norm.reindex(meta_ids).fillna(0.0)

    # Final score
    score = w_text*text_norm + w_emb*emb_norm + w_sent*sent_norm + w_pop*pop

    # remove seeds
    for sid in seed_game_ids:
        score.loc[sid] = -1

    rec_ids = score.sort_values(ascending=False).head(topk).index.astype(int).tolist()

    # Explanation: token overlap with seed union (mechanics/themes/publisher)
    seed_mech = set().union(*[set(mech_tokens.loc[sid]) for sid in seed_game_ids if sid in mech_tokens.index])
    seed_theme = set().union(*[set(theme_tokens.loc[sid]) for sid in seed_game_ids if sid in theme_tokens.index])
    seed_pub = set().union(*[set(pub_tokens.loc[sid]) for sid in seed_game_ids if sid in pub_tokens.index])

    rows = []
    for gid in rec_ids:
        m = set(mech_tokens.loc[gid])
        t = set(theme_tokens.loc[gid])
        p = set(pub_tokens.loc[gid])

        rows.append({
            "game_id": gid,
            "name": df_meta.loc[gid, "name"] if gid in df_meta.index else "",
            "score": float(score.loc[gid]),
            "score_text": float(text_norm.loc[gid]),
            "score_emb": float(emb_norm.loc[gid]),
            "score_sent": float(sent_norm.loc[gid]),
            "score_pop": float(pop.loc[gid]),
            "matched_mechanics": ", ".join(sorted(list(m & seed_mech))[:6]),
            "matched_themes": ", ".join(sorted(list(t & seed_theme))[:6]),
            "matched_publishers": ", ".join(sorted(list(p & seed_pub))[:3]),
        })

    return pd.DataFrame(rows).sort_values("score", ascending=False)

In [9]:
def trait_rank(prefer_mechanics=None, prefer_themes=None, prefer_publishers=None,
               require_all_mechanics=False, require_all_themes=False,
               min_players=None, max_players=None,
               min_playtime=None, max_playtime=None,
               max_complexity=None,
               topk=20,
               w_trait=0.65, w_sent=0.10, w_pop=0.25):
    prefer_mechanics = [s.strip() for s in (prefer_mechanics or []) if str(s).strip()]
    prefer_themes = [s.strip() for s in (prefer_themes or []) if str(s).strip()]
    prefer_publishers = [s.strip() for s in (prefer_publishers or []) if str(s).strip()]

    pref_mech = set(prefer_mechanics)
    pref_theme = set(prefer_themes)
    pref_pub = set(prefer_publishers)

    # Start with all items
    candidates = df_meta.copy()

    # Optional numeric filters if columns exist
    def _apply_range(col, lo=None, hi=None):
        nonlocal candidates
        if col not in candidates.columns:
            return
        s = pd.to_numeric(candidates[col], errors="coerce")
        mask = pd.Series(True, index=candidates.index)
        if lo is not None:
            mask &= (s >= lo)
        if hi is not None:
            mask &= (s <= hi)
        candidates = candidates[mask.fillna(False)]

    # Common column guesses (adjust if needed)
    _apply_range("minplayers", min_players, None)
    _apply_range("maxplayers", None, max_players)
    _apply_range("minplaytime", min_playtime, None)
    _apply_range("maxplaytime", None, max_playtime)
    _apply_range("avgweight", None, max_complexity)  # complexity in BGG often avgweight

    cand_ids = candidates.index.astype(int)

    # Trait match score
    trait_scores = []
    for gid in cand_ids:
        m = set(mech_tokens.loc[gid]) if gid in mech_tokens.index else set()
        t = set(theme_tokens.loc[gid]) if gid in theme_tokens.index else set()
        p = set(pub_tokens.loc[gid]) if gid in pub_tokens.index else set()

        mech_hit = len(m & pref_mech)
        theme_hit = len(t & pref_theme)
        pub_hit = len(p & pref_pub)

        if require_all_mechanics and pref_mech and not pref_mech.issubset(m):
            continue
        if require_all_themes and pref_theme and not pref_theme.issubset(t):
            continue

        denom = (len(pref_mech) + len(pref_theme) + len(pref_pub))
        denom = max(denom, 1)
        score_trait = (mech_hit + theme_hit + pub_hit) / denom

        trait_scores.append((gid, score_trait, mech_hit, theme_hit, pub_hit))

    if not trait_scores:
        return pd.DataFrame(columns=["game_id","name","score"])

    trait_df = pd.DataFrame(trait_scores, columns=["game_id","score_trait","mech_hit","theme_hit","pub_hit"]).set_index("game_id")
    trait_norm = safe_minmax(trait_df["score_trait"], trait_df.index.to_numpy())

    # Sentiment + popularity
    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(trait_df.index).fillna(sent_mean)
        sent_norm = safe_minmax(sent, trait_df.index.to_numpy())
    else:
        sent_norm = pd.Series(0.0, index=trait_df.index)

    pop = pop_norm.reindex(trait_df.index).fillna(0.0)

    final = w_trait*trait_norm + w_sent*sent_norm + w_pop*pop
    top_ids = final.sort_values(ascending=False).head(topk).index.astype(int).tolist()

    rows = []
    for gid in top_ids:
        m = set(mech_tokens.loc[gid]) if gid in mech_tokens.index else set()
        t = set(theme_tokens.loc[gid]) if gid in theme_tokens.index else set()
        p = set(pub_tokens.loc[gid]) if gid in pub_tokens.index else set()

        rows.append({
            "game_id": gid,
            "name": df_meta.loc[gid, "name"] if gid in df_meta.index else "",
            "score": float(final.loc[gid]),
            "score_trait": float(trait_norm.loc[gid]),
            "score_sent": float(sent_norm.loc[gid]),
            "score_pop": float(pop.loc[gid]),
            "matched_mechanics": ", ".join(sorted(list(m & pref_mech))[:6]),
            "matched_themes": ", ".join(sorted(list(t & pref_theme))[:6]),
            "matched_publishers": ", ".join(sorted(list(p & pref_pub))[:3]),
        })

    return pd.DataFrame(rows).sort_values("score", ascending=False)

In [10]:
def discover(seed_game_ids=None,
             prefer_mechanics=None, prefer_themes=None, prefer_publishers=None,
             topk=20):
    seed_game_ids = seed_game_ids or []
    prefer_mechanics = prefer_mechanics or []
    prefer_themes = prefer_themes or []
    prefer_publishers = prefer_publishers or []

    # If no seeds => pure trait ranking
    if len(seed_game_ids) == 0:
        return trait_rank(prefer_mechanics, prefer_themes, prefer_publishers, topk=topk)

    # If seeds exist but traits empty => like-based
    if len(prefer_mechanics)==0 and len(prefer_themes)==0 and len(prefer_publishers)==0:
        return recommend_like(seed_game_ids, topk=topk)

    # Combined: trait filter first (get a candidate pool), then rank by seed score
    cand = trait_rank(prefer_mechanics, prefer_themes, prefer_publishers, topk=300)
    if cand.empty:
        return cand

    like_df = recommend_like(seed_game_ids, topk=1200)

    # merge scores: prefer like_score but keep only trait candidates
    like_df = like_df.set_index("game_id")
    cand = cand.set_index("game_id")

    merged = cand.join(like_df[["score","score_text","score_emb","score_sent","score_pop"]], how="left", rsuffix="_like")
    merged["score_like"] = merged["score"].fillna(0.0)

    # final combined score: 60% like-ranking + 40% trait score
    merged["score_final"] = 0.60*merged["score_like"] + 0.40*merged["score_trait"]

    out = merged.sort_values("score_final", ascending=False).head(topk).reset_index()
    out = out.rename(columns={"score_final":"score"})
    return out

In [11]:
# Example A: Find games like (seed-based)
recommend_like([13], topk=10)  # e.g., seed game id 13

# Example B: Find games with traits
trait_rank(
    prefer_mechanics=["Cooperative Game"],
    prefer_themes=["Medical"],
    prefer_publishers=[],
    topk=10
)

# Example C: Combined (traits + seeds)
discover(
    seed_game_ids=[13],
    prefer_mechanics=["Cooperative Game"],
    prefer_themes=["Medical"],
    topk=10
)

,game_id,name,score,score_trait,score_sent,score_pop,matched_mechanics,matched_themes,matched_publishers,score_like,score_text,score_emb,score_sent_like,score_pop_like,score
0,30549,Pandemic,0.970406,1.0,0.704058,1.000000,Cooperative Game,Medical,,0.970406,0.0,0.906982,0.704058,1.000000,0.982243
1,161936,Pandemic Legacy: Season 1,0.803887,1.0,0.509526,0.411739,Cooperative Game,Medical,,0.803887,0.0,0.000000,0.509526,0.411739,0.882332
2,221107,Pandemic Legacy: Season 2,0.731991,1.0,0.509526,0.124156,Cooperative Game,Medical,,0.731991,0.0,0.000000,0.509526,0.124156,0.839195
3,150658,Pandemic: The Cure,0.723621,1.0,0.509526,0.090674,Cooperative Game,Medical,,0.723621,0.0,0.000000,0.509526,0.090674,0.834173
4,198928,Pandemic: Iberia,0.721468,1.0,0.509526,0.082062,Cooperative Game,Medical,,0.721468,0.0,0.000000,0.509526,0.082062,0.832881
5,280789,Pandemic: Rapid Response,0.705762,1.0,0.509526,0.019237,Cooperative Game,Medical,,0.705762,NaN,NaN,NaN,NaN,0.823457
6,216597,Flatline,0.703999,1.0,0.509526,0.012184,Cooperative Game,Medical,,0.703999,NaN,NaN,NaN,NaN,0.822399
7,301919,Pandemic: Hot Zone – North America,0.703734,1.0,0.509526,0.011127,Cooperative Game,Medical,,0.703734,NaN,NaN,NaN,NaN,0.822241
8,231197,Raxxon,0.703268,1.0,0.509526,0.009261,Cooperative Game,Medical,,0.703268,NaN,NaN,NaN,NaN,0.821961
9,245444,Holding On: The Troubled Life of Billy Kerr,0.702945,1.0,0.509526,0.007971,Cooperative Game,Medical,,0.702945,NaN,NaN,NaN,NaN,0.821767
